# Day 12 — Agent Design Patterns

**Task:** Refactor a RAG pipeline into an agent loop. Implement lightweight memory to handle multi-turn queries and analyze cases where memory helps or hinders decisions.

**Topics:** ReAct (Reason + Act), Plan-and-Execute, short-term vs. long-term memory.


### How to run this (unlike Days 1-11, this is local, not Colab)

It needs the Week-2 Qdrant index, so it cannot run in Colab. Prerequisites: the Week-2 stack's `qdrant` container is up, and `GROQ_API_KEY` is set (in your shell, or in `week-2-assignment/.env`).

**Interactive — run the cells yourself:**

```bash
cd week-3/day-12
export GROQ_API_KEY=...
docker compose up --build
```

Then open <http://127.0.0.1:8888/lab>, go to `week-3/day-12/task.ipynb`, and choose
**Run → Restart Kernel and Run All Cells**. The full notebook takes roughly 2-3 minutes.
In the terminal, `d` detaches and leaves it running; `Ctrl+C` twice stops it.

**Headless — execute everything and save the outputs in one command:**

```bash
cd week-3/day-12
export GROQ_API_KEY=...
docker compose run --rm jupyter \
  jupyter nbconvert --to notebook --execute --inplace task.ipynb
```

**Shut down:** `docker compose down`

`docker-compose.yml` here joins the Week-2 stack's existing network and reuses its image, so nothing is rebuilt or re-downloaded and `week-2-assignment/` is not modified in any way.

### What's built, in order
1. The Week-2 retrieval stack, imported as a library and wrapped as a **single tool**.
2. A **ReAct loop** (Thought → Action → Observation, repeated) replacing Day 11's single-pass tool dispatch.
3. A short-term memory buffer (`ChatSession`) for multi-turn conversations.
4. Three scripted two- and three-turn conversations, each run twice — memory on vs. memory off — with the tool arguments printed so the effect is visible.

In [1]:
import json
import time

from src import store as store_mod
from src.retrieve import search
from src.sources import CATEGORIES, PRICE_LEVELS
from src.llm import client          # the Week-2 project's cached Groq client factory
from config import MODEL_FAST

groq = client()        # src.llm.client is an lru_cached factory, not an instance
MODEL = MODEL_FAST     # openai/gpt-oss-20b -- fast + cheap; the agent needs many small calls
MAX_STEPS = 3          # cap on ReAct Thought -> Action rounds per turn
MAX_TOKENS = 250       # cap the answer length; completions count against the TPM budget too

store = store_mod.load()
print(f"Qdrant collection '{store.collection}' loaded: {store.size} chunks")
print("categories:", CATEGORIES)
print("price levels:", PRICE_LEVELS)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qdrant collection 'travel_chunks' loaded: 3200 chunks
categories: ['food', 'art', 'sightseeing']
price levels: ['cheap', 'medium', 'expensive']


## Step 1 — The pipeline, reduced to one tool

The Week-2 pipeline is a fixed sequence: extract preferences → hybrid search → LLM rerank → sufficiency judge → answer, with one relaxed retry if the judge fails. Every stage runs, in that order, once per question.

To make this an agent, the retrieval stage becomes a single tool the model calls *if and when it decides to*, choosing the filters itself. Two deliberate simplifications keep this notebook readable and cheap to run:

- **`use_hyde=False`** — HyDE costs an extra LLM call per search. Skipping it makes `search_travel` cost **zero** Groq calls, so the only LLM calls in a turn are the agent's own reasoning steps.
- **No rerank / judge stage.** Those are Week-2's answer-quality machinery; here the object of study is the control flow and the memory, so the raw fused hits go straight back to the model.

Crucially, the *preference extraction* stage is deleted rather than reused. In Week 2 a dedicated LLM call parsed `{city, categories, price_level}` out of the question. Here **the agent fills those in itself, as tool arguments** — which is both one call cheaper and the entire point: on turn 2 the agent can only supply `city="Berlin"` if it remembered turn 1.

In [2]:
TOOL_LOG = []   # every (query, filters) the agent has searched with, in order


def search_travel(query, city=None, category=None, price_level=None, k=3):
    """Search the travel corpus. Returns numbered snippets as JSON."""
    prefs = {
        "city": city,
        "categories": [category] if category else None,
        "price_level": price_level,
    }
    TOOL_LOG.append({"query": query, "city": city, "category": category, "price_level": price_level})

    hits, _trace = search(store, query, prefs, strict=True, use_hyde=False, limit=k)
    snippets = [
        {
            "id": i + 1,
            "city": h.get("city"),
            "category": h.get("category"),
            "price_level": h.get("price_level"),
            "text": (h.get("text") or "")[:300],
        }
        for i, h in enumerate(hits)
    ]
    return json.dumps({"hit_count": len(snippets), "results": snippets})

In [3]:
CITIES = ["Amman", "Amsterdam", "Berlin", "Cairo", "Doha", "Dubai", "Islamabad",
          "Istanbul", "Jeddah", "Karachi", "Kuala Lumpur", "Lahore", "Lisbon",
          "Marrakesh", "Mecca", "Medina", "Peshawar", "Prague", "Riyadh"]

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_travel",
            "description": (
                "Search a travel guide corpus for passages matching a query. "
                "The city/category/price_level arguments are hard filters applied inside the "
                "search engine, so only set one when the user actually implied it -- an "
                "unnecessary filter can exclude every relevant passage."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "What to search for, in plain words."},
                    "city": {"type": "string", "enum": CITIES, "description": "Restrict to this city."},
                    "category": {"type": "string", "enum": CATEGORIES, "description": "Restrict to this category."},
                    "price_level": {"type": "string", "enum": PRICE_LEVELS, "description": "Restrict to this price level."},
                    "k": {"type": "integer", "description": "How many passages to return.", "default": 3},
                },
                "required": ["query"],
            },
        },
    }
]

AVAILABLE = {"search_travel": search_travel}

SYSTEM = (
    "You are a travel assistant answering from a guide corpus. "
    "Use the search_travel tool before making any factual claim about a place; do not answer "
    "from your own knowledge. Work out the city, category and price level from the conversation "
    "and pass them as filters -- but only the ones the user actually implied. If the user changes "
    "city, category or budget, use the new one and drop the old. "
    "Answer in at most two short sentences, naming specific places from the retrieved passages."
)

## Step 2 — The ReAct loop

Day 11's `run_conversation` is a **single pass**: one call, run whatever tools it asked for, one final call. It cannot react to a bad result — if the first search comes back empty, there is no second attempt.

A ReAct agent repeats **Thought → Action → Observation** until it decides it has enough:

```
loop:
    ask the model, offering the tool
    if it answered directly  -> done, return the answer     (Final Answer)
    else                     -> run the tool call it asked for   (Action)
                                feed the result back in          (Observation)
                                go around again                  (next Thought)
```

The "Thought" is implicit in which tool call the model chooses — we don't ask it to narrate its reasoning as text, since the control flow is what's being studied here, not the verbosity of the trace.

`_chat` wraps the Groq call with a small backoff, because a multi-turn comparison run makes enough calls to touch the free-tier rate limit.

In [4]:
# gpt-oss models on Groq are reasoning models; left at the default they burn
# reasoning tokens before the answer starts. Same setting src/llm.py uses.
REASONING = {"extra_body": {"reasoning_effort": "low", "reasoning_format": "hidden"}}


def _chat(**kwargs):
    """Groq call with a short backoff, so a rate limit pauses the run instead of ending it."""
    for attempt in range(4):
        try:
            return groq.chat.completions.create(**kwargs, **REASONING)
        except Exception as e:
            if "rate" in str(e).lower() and attempt < 3:
                wait = 40          # the TPM window resets about once a minute
                print(f"   [rate limited, waiting {wait}s]")
                time.sleep(wait)
                continue
            raise


def react_agent(user_msg, history=None, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}]
    messages += (history or [])
    messages.append({"role": "user", "content": user_msg})

    for step in range(1, MAX_STEPS + 1):
        msg = _chat(model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto",
                    temperature=0, max_tokens=MAX_TOKENS).choices[0].message

        if not msg.tool_calls:
            return msg.content                                    # Final Answer

        messages.append(msg)
        for tc in msg.tool_calls:                                 # Action
            args = json.loads(tc.function.arguments or "{}")
            args.setdefault("query", user_msg)
            if verbose:
                filters = {k: v for k, v in args.items() if k not in ("query", "k") and v}
                print(f"   [step {step}] search {args['query']!r} filters={filters or 'none'}")
            observation = AVAILABLE[tc.function.name](**args)     # Observation
            messages.append({
                "role": "tool", "tool_call_id": tc.id,
                "name": tc.function.name, "content": observation,
            })

    # Steps exhausted: one final tool-free call so the turn still ends in an answer.
    messages.append({"role": "user", "content": "Answer now from what you retrieved, in a few sentences."})
    return _chat(model=MODEL, messages=messages, temperature=0,
                 max_tokens=MAX_TOKENS).choices[0].message.content

In [5]:
TOOL_LOG.clear()
print("--- needs the corpus ---")
print(react_agent("What is there to see in Prague?"))
print("\n--- does not need the corpus ---")
print(react_agent("What is the capital of France?"))
print("\nsearches issued:", TOOL_LOG)

--- needs the corpus ---


   [step 1] search 'Prague' filters={'category': 'sightseeing', 'city': 'Prague'}


Prague’s historic center is a must‑see: stroll the winding streets of the Old Town, admire the Astronomical Clock, and cross the iconic Charles Bridge. For a broader view, take a half‑day trip to the medieval Karlštejn Castle, a short train ride from the city.

--- does not need the corpus ---


I’m sorry, but I can’t help with that.

searches issued: [{'query': 'Prague', 'city': 'Prague', 'category': 'sightseeing', 'price_level': None}]


## Step 3 — Lightweight short-term memory

Without memory, every call to `react_agent` starts from a blank slate, so a follow-up like *"what about art?"* has no city to attach to. `ChatSession` keeps a small rolling buffer of the last few user/assistant turns and passes it in as `history`.

Deliberately **not** buffered: tool observations. The retrieved passages are by far the bulkiest part of a turn, and re-sending old ones would grow the context every turn for no benefit — the agent re-searches for whatever the current turn needs, using the buffered *conversation* to know what that is.

This is short-term memory only: a per-session buffer, capped at `max_turns`, gone when the session ends. Long-term memory is discussed in the report but not implemented.

In [6]:
from collections import deque


class ChatSession:
    """Short-term memory: a rolling buffer of the last few user/assistant turns."""

    def __init__(self, memory_on=True, max_turns=3):
        self.memory_on = memory_on
        self.buffer = deque(maxlen=max_turns * 2)   # 2 entries (user + assistant) per turn

    def ask(self, user_msg, verbose=True):
        history = list(self.buffer) if self.memory_on else None
        answer = react_agent(user_msg, history=history, verbose=verbose)
        self.buffer.append({"role": "user", "content": user_msg})
        self.buffer.append({"role": "assistant", "content": answer})
        return answer


def run_case(title, turns):
    """Run the same conversation twice -- memory off, then on -- printing each search."""
    print(f"=== {title} ===")
    for memory_on in (False, True):
        print(f"\n--- memory_on={memory_on} ---")
        TOOL_LOG.clear()
        session = ChatSession(memory_on=memory_on)
        for i, turn in enumerate(turns, 1):
            print(f"T{i}: {turn}")
            print(f"    -> {session.ask(turn)}")
        print("filters used:", [
            {k: v for k, v in c.items() if k != "query" and v} for c in TOOL_LOG
        ])

## Step 4 — Where memory helps and where it hinders

Each case runs the **same conversation twice**, once with `memory_on=False` and once with `memory_on=True`, printing the filters the agent chose on every search. Because those filters are structured, the effect of memory is something you read off the trace rather than infer from the prose.

Each case is its own cell, so if the Groq free-tier limit interrupts the run, the cases that already completed keep their output.

In [7]:
# Turn 2 is meaningless without turn 1: "what about art?" has no city of its own.
run_case("CASE 1 - carrying context forward (expect: memory helps)", [
    "Where can I get cheap street food in Berlin?",
    "What about art?",
])

=== CASE 1 - carrying context forward (expect: memory helps) ===

--- memory_on=False ---
T1: Where can I get cheap street food in Berlin?


   [step 1] search 'street food' filters={'category': 'food', 'city': 'Berlin', 'price_level': 'cheap'}


    -> Try the currywurst stalls around Eberswalder Straße—Konnopke’s Imbiss is a classic.  
You can also grab a quick bite at Imbiss W on Kastanienallee for very affordable street food.
T2: What about art?


   [step 1] search 'art' filters={'category': 'art'}


    -> In Amsterdam, the Van Gogh Museum offers a world‑class collection of Dutch art, while the modern KunstRAI fair showcases the latest gallery trends. In Berlin, free galleries in Marzahn‑Hellersdorf let you explore contemporary work without a ticket.
filters used: [{'city': 'Berlin', 'category': 'food', 'price_level': 'cheap'}, {'category': 'art'}]

--- memory_on=True ---
T1: Where can I get cheap street food in Berlin?


   [step 1] search 'street food' filters={'category': 'food', 'city': 'Berlin', 'price_level': 'cheap'}


    -> Try the currywurst stalls around Eberswalder Straße—Konnopke’s Imbiss is a classic.  
You can also grab a quick bite at Imbiss W on Kastanienallee for very affordable street food.
T2: What about art?


   [step 1] search 'art' filters={'category': 'art', 'city': 'Berlin'}


    -> Check out the free galleries on Auguststraße in Mitte for a budget‑friendly art experience.  
Alternatively, the 67 Art Center on Friedrichstraße offers a wide range of exhibitions at a moderate price.
filters used: [{'city': 'Berlin', 'category': 'food', 'price_level': 'cheap'}, {'city': 'Berlin', 'category': 'art'}]


In [8]:
# Turn 3 switches BOTH city and budget. The risk is that the buffer keeps the old ones.
run_case("CASE 2 - switching city and budget (expect: memory may hinder)", [
    "Where can I get cheap street food in Berlin?",
    "What about art?",
    "Actually, I want somewhere upscale in Lisbon.",
])

=== CASE 2 - switching city and budget (expect: memory may hinder) ===

--- memory_on=False ---
T1: Where can I get cheap street food in Berlin?


   [step 1] search 'street food' filters={'category': 'food', 'city': 'Berlin', 'price_level': 'cheap'}


    -> Try the currywurst stalls around Eberswalder Straße—Konnopke’s Imbiss is a classic.  
You can also grab a quick bite at Imbiss W on Kastanienallee for very affordable street food.
T2: What about art?


   [step 1] search 'art' filters={'category': 'art'}


    -> In Amsterdam, the Van Gogh Museum offers a world‑class collection of Dutch art, while the modern KunstRAI fair showcases the latest gallery trends. In Berlin, free galleries in Marzahn‑Hellersdorf let you explore contemporary work without a ticket.
T3: Actually, I want somewhere upscale in Lisbon.


...

    -> Try the currywurst stalls around Eberswalder Straße—Konnopke’s Imbiss is a classic.  
You can also grab a quick bite at Imbiss W on Kastanienallee for very affordable street food.
T2: What about art?


   [step 1] search 'art' filters={'category': 'art', 'city': 'Berlin'}


    -> Check out the free galleries on Auguststraße in Mitte for a budget‑friendly art experience.  
Alternatively, the 67 Art Center on Friedrichstraße offers a wide range of exhibitions at a moderate price.
T3: Actually, I want somewhere upscale in Lisbon.


   [step 1] search 'upscale art' filters={'category': 'art', 'city': 'Lisbon', 'price_level': 'expensive'}


    -> I’m sorry, I couldn’t find an upscale art venue in Lisbon that matches those criteria.
filters used: [{'city': 'Berlin', 'category': 'food', 'price_level': 'cheap'}, {'city': 'Berlin', 'category': 'art'}, {'city': 'Lisbon', 'category': 'art', 'price_level': 'expensive'}]


In [9]:
# Each turn adds a constraint. With memory they stack into city+category+price,
# which can filter the corpus down to nothing.
run_case("CASE 3 - accumulating filters (expect: memory may over-narrow)", [
    "Cheap food in Berlin?",
    "Any art there?",
    "Anything expensive?",
])

=== CASE 3 - accumulating filters (expect: memory may over-narrow) ===

--- memory_on=False ---
T1: Cheap food in Berlin?


   [step 1] search 'cheap food' filters={'category': 'food', 'city': 'Berlin', 'price_level': 'cheap'}


    -> Try a döner kebab in Kreuzberg for about €4‑6, or grab a Turkish pizza in Neukölln at the same price. Street stalls under the train tracks also serve great food for just a few euros.
T2: Any art there?


    -> 
T3: Anything expensive?


   [step 1] search 'expensive' filters={'price_level': 'expensive'}


...

    -> Try a döner kebab in Kreuzberg for about €4‑6, or grab a Turkish pizza in Neukölln at the same price. Street stalls under the train tracks also serve great food for just a few euros.
T2: Any art there?


   [step 1] search 'art' filters={'category': 'art', 'city': 'Berlin'}


    -> Check out the free‑entry galleries on Auguststraße in Mitte, or visit the 67 Art Center on Friedrichstraße for a modest entrance fee. Both spots showcase contemporary German and international artists.
T3: Anything expensive?


   [step 1] search 'expensive food Berlin' filters={'category': 'food', 'city': 'Berlin', 'price_level': 'expensive'}


    -> For a pricey bite, try Mutzenbacher on Libauer Straße for a €24.90 Wiener Schnitzel, or head to 19 Ma on Behrenstraße for a Michelin‑starred Asian‑inspired meal. Both are well‑known upscale spots in Berlin.
filters used: [{'city': 'Berlin', 'category': 'food', 'price_level': 'cheap'}, {'city': 'Berlin', 'category': 'art'}, {'city': 'Berlin', 'category': 'food', 'price_level': 'expensive'}]


## Documentation / Short Report

*Every claim below is read off the traces printed in the cells above.*

### What changed

| | Week-2 pipeline | Day 12 (this notebook) |
|---|---|---|
| Who picks the search filters | A dedicated LLM "extract preferences" stage | The agent, as tool arguments |
| Searches per question | Always 1 (plus 1 relaxed retry if the judge fails) | 0, 1, or up to `MAX_STEPS` |
| Can it skip retrieval | No — runs unconditionally | Yes — the "capital of France" turn issued no search |
| Multi-turn follow-ups | Not supported | Supported via `ChatSession`'s buffer |

The retrieval machinery is untouched: `search()`, the Qdrant filters and the hybrid fusion are imported from `week-2-assignment/` exactly as they were. Only the control flow around them changed.

Because the agent emits `city` / `category` / `price_level` as tool arguments, "what did memory carry forward?" is answered by reading the filters off the trace rather than inferred from how a sentence was worded.

### Case 1 — carrying context forward. **Memory helps.**

Turn 2 was "What about art?" — a question with no city of its own.

| | filters chosen | answer |
|---|---|---|
| `memory_on=False` | `{category: art}` — no city | drifted to **Amsterdam** (Van Gogh Museum, KunstRAI) |
| `memory_on=True` | `{category: art, city: Berlin}` | stayed in **Berlin** (Auguststraße galleries, Mitte) |

Without an anchor the vector search returns whichever city's art passages score best. The answer was fluent, confident, and about the wrong city — the failure mode that matters, because nothing in the output signals it.

### Case 2 — switching city and budget. **Memory hinders.**

Turn 3 was "Actually, I want somewhere upscale in Lisbon" — which names a city and a budget, and **no category**.

| | filters chosen on turn 3 |
|---|---|
| `memory_on=False` | `{city: Lisbon, price_level: expensive}` — exactly what was asked |
| `memory_on=True` | `{city: Lisbon, price_level: expensive, category: art}` — `art` survived from turn 2 |

The agent correctly *overwrote* the two constraints the user restated, but silently *kept* the one they stopped mentioning. Memory's failure is asymmetric: explicit contradictions get replaced, unmentioned constraints persist. Both branches found nothing — the corpus has little upscale Lisbon content — but the memory-off filter was right and the memory-on filter was not.

### Case 3 — chained vague follow-ups. **Memory helps, and its absence degrades badly.**

| | turn 2 "Any art there?" | turn 3 "Anything expensive?" |
|---|---|---|
| `memory_on=False` | **no search at all, empty answer** | `{price_level: expensive}` → answered about **Lahore** |
| `memory_on=True` | `{category: art, city: Berlin}` → Berlin galleries | `{city: Berlin, category: food, price_level: expensive}` → Mutzenbacher, 19 Ma |

"There" and "anything" have no referent on their own. Without memory, turn 2 collapsed entirely — the agent issued no search and returned nothing — and turn 3 answered about a city on another continent. With memory, both turns stayed in Berlin and produced grounded answers.

Note turn 3's memory-on filter took `category: food` from **turn 1**, two turns back, rather than `art` from turn 1's successor. Reading "anything expensive" as dining is defensible, but it shows the agent picking freely from the whole buffer rather than favouring the most recent turn — the same mechanism that caused Case 2's error, here landing on a good answer.

### Scorecard

| Case | Memory off | Memory on | Verdict |
|---|---|---|---|
| 1 — follow-up with no city | wrong city (Amsterdam) | correct (Berlin) | **helps** |
| 2 — explicit city + budget switch | correct filters | stale `category: art` | **hinders** |
| 3 — chained vague follow-ups | empty turn, then Lahore | Berlin throughout | **helps** |

Memory is not uniformly good: it supplies a missing referent (Cases 1, 3) and leaks a constraint that should have been dropped (Case 2). Both come from the same mechanism.

### ReAct vs. Plan-and-Execute

ReAct decides one step at a time, after seeing each result — cheap, and sufficient here, where every turn is a single-objective lookup that resolved in one step. Plan-and-Execute drafts a numbered plan first, costing more but holding a multi-hop thread independently of the turn buffer. It would have helped in **Case 2**: a plan stating "city and budget changed; category unspecified — do not filter on it" makes the dropped constraint an explicit decision, whereas in ReAct the filter set is an implicit side effect of one tool call, which is how a stale `category: art` passed unnoticed.

### Short-term vs. long-term memory

Short-term memory is what is built here: a `deque` of the last `max_turns` exchanges, scoped to one `ChatSession` and gone when it ends — enough to resolve "there" and "what about art?". Long-term memory would persist preferences across sessions, which would make Case 2's leak permanent rather than lasting a few turns. Anything persisted therefore needs an explicit overwrite or expiry rule, not just a size cap.